# 🐍 Manejo de Archivos con Python
### PyLadies Cuernavaca — Python desde cero : 4 días de fundamentos.

**Instructora(s):**
Dra. Maricela Carrera

M.O.C.A. Edna Cruz F.

Dra. Alida Zárate

M. en C. Wendolyn Estrada

**Dataset de trabajo:** [Palmer Penguins](https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data) (Kaggle) y archivo FASTA de ejemplo.

En esta sesión aprenderemos a abrir, leer, escribir y manejar errores con archivos en Python, usando datos reales de mediciones de pingüinos como hilo conductor.

**Contenido:**
1. Formas de tener un archivo en Google Colab
2. La función `open()` y sus modos
3. Leer archivos: `read()`, `readline()`, `readlines()`
4. Cerrar archivos: `close()`
5. Método recomendado: `with` statement
6. Manejo de errores: `try` / `except`
7. Escribir archivos: modo `w`, modo `a`, `write()`, `writelines()`
8. Escribir archivos a partir de diccionarios


## Formas de tener un archivo en Google Colab

Antes de poder abrir un archivo con Python, el archivo tiene que *existir* en algún lugar al que Colab pueda acceder. Hay varias formas de lograrlo:

* **Subir el archivo directamente a la sesión:** rápido y simple, pero el archivo **se borra** cuando termina la sesión de Colab.
* **Montar Google Drive:** el archivo queda disponible en `/content/drive/MyDrive/...` y **persiste** entre sesiones. Es la forma recomendada para trabajar con datasets que usaremos varias veces.
* **Descargar desde una URL:** útil cuando el archivo vive en internet (por ejemplo, un CSV público).
* **Desde un script `.py` (fuera de Colab):** no existen `files.upload()` ni `drive.mount()` — el archivo simplemente vive en el sistema de archivos de tu computadora, y lo referencias con una ruta relativa (`"penguins.csv"`) o absoluta (`"/home/usuario/datos/penguins.csv"`).

Vamos a probar las dos primeras, que son las que usaremos en el taller.

In [2]:
# Opción A: subir el archivo directo a esta sesión de Colab
# Se abrirá un cuadro de diálogo para elegir el archivo desde tu computadora
from google.colab import files

uploaded = files.upload()

Saving penguins.csv to penguins.csv
Saving secuencias_pyladiescuerna.fasta to secuencias_pyladiescuerna.fasta


> ***Nota :*** Con `files.upload()` el archivo vive solo en esta sesión. Si se reinicia el entorno de ejecución, hay que volver a subirlo.

In [ ]:
# Opción B: montar Google Drive (persistente)
from google.colab import drive

drive.mount('/content/drive')

# Tip: puedes dar clic derecho sobre el archivo en el panel izquierdo
# y elegir "Copiar ruta" para obtener la ruta exacta

Mounted at /content/drive


In [3]:
## Ejercicio: define la ruta a tu archivo penguins.csv
## (usa la ruta de Drive si lo montaste, o solo "penguins.csv" si lo subiste directo)
## Ruta en carpeta de Drive: /content/drive/MyDrive/Colab Notebooks/Datasets/penguins.csv

ruta_archivo = "/content/penguins.csv"
print(type(ruta_archivo))
#ruta_archivo = "/content/drive/MyDrive/Colab Notebooks/Datasets/penguins.csv"


<class 'str'>


# 🐧 Dataset: Palmer Penguins (Pingüinos de Palmer)


El dataset **Palmer Penguins** es una alternativa moderna y amigable al clásico dataset *Iris*. Fue recopilado por la **Dra. Kristen Gorman** y la Estación Palmer en la Antártida, y contiene mediciones anatómicas de **344 pingüinos**.

---

### Especificaciones del Dataset

* **Muestras totales:** 344 pingüinos.
* **Especies incluidas (3):** Adelie, Chinstrap (Barbijo) y Gentoo Juanito).
* **Ubicación:** 3 islas del Archipiélago Palmer (Antártida): Torgersen, Biscoe y Dream.


### Variables / Características (Features)

El dataset cuenta con variables numéricas y categóricas para cada pingüino:

1. **species:** Especie del pingüino (Adelie, Chinstrap, Gentoo).
2. **island:** Isla donde se registró la muestra (Biscoe, Dream, Torgersen).
3. **bill_length_mm:** Longitud del culmen / pico (en milímetros).
4. **bill_depth_mm:** Profundidad del culmen / pico (en milímetros).
5. **flipper_length_mm:** Longitud de la aleta (en milímetros).
6. **body_mass_g:** Masa corporal / peso (en gramos).
7. **sex:** Sexo del pingüino (MALE, FEMALE).
8. **year:** Año de la recolección del dato (2007, 2008, 2009).

---


## La función `open()` y sus modos

Para abrir un archivo en Python se utiliza la función `open()`. Recibe dos argumentos: el nombre (o ruta) del archivo, y el **modo** en el que se va a abrir.

```python
archivo = open("nombre_del_archivo.txt", "r")
```

| Modo | Significado |
|------|-------------|
| `"r"` | Lectura (*read*). El archivo debe existir. |
| `"w"` | Escritura (*write*). Crea el archivo si no existe, **sobrescribe** si ya existe. |
| `"a"` | Añadir (*append*). Agrega contenido al final sin borrar lo que ya había. |
| `"b"` | Modo binario, se combina con los anteriores (ej. `"rb"`) para archivos no-texto. |


In [4]:
## Ejercicio: abre penguins.csv en modo lectura
archivo = open(ruta_archivo, "r")

print(type(archivo))

<class '_io.TextIOWrapper'>


In [5]:
print(archivo)

<_io.TextIOWrapper name='/content/penguins.csv' mode='r' encoding='utf-8'>


## Leer archivos

Podemos leer el contenido de un archivo de tres formas distintas:

* **`read()`**: lee **todo** el archivo de una vez y lo regresa como un solo string.
* **`readline()`**: lee **una línea** cada vez que se llama. Útil en bucles.
* **`readlines()`**: lee **todas las líneas** y las regresa como una lista de strings.

Vamos a probar los tres métodos sobre `penguins.csv` y comparar qué regresa cada uno.

In [6]:
archivo = open(ruta_archivo, "r")

# read(): todo el archivo como un solo string
contenido = archivo.read()
print(contenido[:300])  # primeros 300 caracteres
print(type(contenido))

archivo.close()

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
Adelie,Torgersen,39.1,18.7,181,3750,male,2007
Adelie,Torgersen,39.5,17.4,186,3800,female,2007
Adelie,Torgersen,40.3,18,195,3250,female,2007
Adelie,Torgersen,NA,NA,NA,NA,NA,2007
Adelie,Torgersen,36.7,19.3,193,3450,fema
<class 'str'>


In [7]:
print(contenido)

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
Adelie,Torgersen,39.1,18.7,181,3750,male,2007
Adelie,Torgersen,39.5,17.4,186,3800,female,2007
Adelie,Torgersen,40.3,18,195,3250,female,2007
Adelie,Torgersen,NA,NA,NA,NA,NA,2007
Adelie,Torgersen,36.7,19.3,193,3450,female,2007
Adelie,Torgersen,39.3,20.6,190,3650,male,2007
Adelie,Torgersen,38.9,17.8,181,3625,female,2007
Adelie,Torgersen,39.2,19.6,195,4675,male,2007
Adelie,Torgersen,34.1,18.1,193,3475,NA,2007
Adelie,Torgersen,42,20.2,190,4250,NA,2007
Adelie,Torgersen,37.8,17.1,186,3300,NA,2007
Adelie,Torgersen,37.8,17.3,180,3700,NA,2007
Adelie,Torgersen,41.1,17.6,182,3200,female,2007
Adelie,Torgersen,38.6,21.2,191,3800,male,2007
Adelie,Torgersen,34.6,21.1,198,4400,male,2007
Adelie,Torgersen,36.6,17.8,185,3700,female,2007
Adelie,Torgersen,38.7,19,195,3450,female,2007
Adelie,Torgersen,42.5,20.7,197,4500,male,2007
Adelie,Torgersen,34.4,18.4,184,3325,female,2007
Adelie,Torgersen,46,21.5,194,4200,male,2007
Adelie

In [9]:
archivo = open(ruta_archivo, "r")

# readline(): una línea a la vez
encabezado = archivo.readline().strip()
primer_pinguino = archivo.readline().strip()

print("Encabezado:", encabezado)
print("Primer registro:", primer_pinguino)

archivo.close()

Encabezado: species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
Primer registro: Adelie,Torgersen,39.1,18.7,181,3750,male,2007


In [10]:
archivo = open(ruta_archivo, "r")

# readlines(): todas las líneas como una lista
lineas = archivo.readlines()

print(f"Número total de pingüinos registrados: {len(lineas) - 1}")  # -1 por el encabezado
print("Encabezado:", lineas[0])
print("Primer pingüino:", lineas[1])
print("Último pingüino:", lineas[-1])

archivo.close()

Número total de pingüinos registrados: 344
Encabezado: species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year

Primer pingüino: Adelie,Torgersen,39.1,18.7,181,3750,male,2007

Último pingüino: Chinstrap,Dream,50.2,18.7,198,3775,female,2009



In [11]:
print(lineas)

['species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year\n', 'Adelie,Torgersen,39.1,18.7,181,3750,male,2007\n', 'Adelie,Torgersen,39.5,17.4,186,3800,female,2007\n', 'Adelie,Torgersen,40.3,18,195,3250,female,2007\n', 'Adelie,Torgersen,NA,NA,NA,NA,NA,2007\n', 'Adelie,Torgersen,36.7,19.3,193,3450,female,2007\n', 'Adelie,Torgersen,39.3,20.6,190,3650,male,2007\n', 'Adelie,Torgersen,38.9,17.8,181,3625,female,2007\n', 'Adelie,Torgersen,39.2,19.6,195,4675,male,2007\n', 'Adelie,Torgersen,34.1,18.1,193,3475,NA,2007\n', 'Adelie,Torgersen,42,20.2,190,4250,NA,2007\n', 'Adelie,Torgersen,37.8,17.1,186,3300,NA,2007\n', 'Adelie,Torgersen,37.8,17.3,180,3700,NA,2007\n', 'Adelie,Torgersen,41.1,17.6,182,3200,female,2007\n', 'Adelie,Torgersen,38.6,21.2,191,3800,male,2007\n', 'Adelie,Torgersen,34.6,21.1,198,4400,male,2007\n', 'Adelie,Torgersen,36.6,17.8,185,3700,female,2007\n', 'Adelie,Torgersen,38.7,19,195,3450,female,2007\n', 'Adelie,Torgersen,42.5,20.7,197,4500,male,2007\n', 'A

In [ ]:
## Ejercicio: recorre 'lineas' (sin el encabezado) e imprime
## únicamente la especie y la isla de cada pingüino.
## Tip: cada línea es un string separado por comas -> usa .split(",")

# readlines(): todas las líneas como una lista
archivo = open(ruta_archivo, "r")
lineas = archivo.readlines()

for linea in lineas[1:]:
  fila = linea.strip().split(",")
  #print(fila)
  especie = fila[0]
  isla = fila[1]
  print(f"Especie: {especie}, Isla: {isla}")

archivo.close()

## Cerrar archivos: `close()`

Al terminar de leer o escribir un archivo, es importante **cerrarlo** con `close()`. Esto libera recursos del sistema y, en el caso de escritura, asegura que todo lo escrito se guarde realmente en disco.

```python
archivo.close()
```

Si no cerramos el archivo, podemos tener comportamientos inesperados: datos que no se terminan de guardar, o el archivo bloqueado para otros procesos.

In [16]:
archivo = open(ruta_archivo, "r")

print("¿Está cerrado?", archivo.closed)  # False

archivo.close()

print("¿Está cerrado?", archivo.closed)  # True

¿Está cerrado? False
¿Está cerrado? True


## Método recomendado: `with` statement

Acordarnos de llamar `close()` cada vez, es fácil de olvidar. La forma recomendada de trabajar con archivos en Python es usando `with`, que **cierra el archivo automáticamente** al salir del bloque, incluso si ocurre un error en el camino.

```python
with open("archivo.txt", "r") as archivo:
    contenido = archivo.read()
# aquí afuera el archivo ya está cerrado, sin necesidad de escribir close()
```

Vamos a reescribir el ejercicio de la sección 1.8.3 usando `with`.

In [20]:
with open(ruta_archivo, "r") as archivo:
    lineas = archivo.readlines()


print(f"¿Está cerrado el archivo fuera del bloque with? {archivo.closed}")
print(lineas)

¿Está cerrado el archivo fuera del bloque with? True
['species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year\n', 'Adelie,Torgersen,39.1,18.7,181,3750,male,2007\n', 'Adelie,Torgersen,39.5,17.4,186,3800,female,2007\n', 'Adelie,Torgersen,40.3,18,195,3250,female,2007\n', 'Adelie,Torgersen,NA,NA,NA,NA,NA,2007\n', 'Adelie,Torgersen,36.7,19.3,193,3450,female,2007\n', 'Adelie,Torgersen,39.3,20.6,190,3650,male,2007\n', 'Adelie,Torgersen,38.9,17.8,181,3625,female,2007\n', 'Adelie,Torgersen,39.2,19.6,195,4675,male,2007\n', 'Adelie,Torgersen,34.1,18.1,193,3475,NA,2007\n', 'Adelie,Torgersen,42,20.2,190,4250,NA,2007\n', 'Adelie,Torgersen,37.8,17.1,186,3300,NA,2007\n', 'Adelie,Torgersen,37.8,17.3,180,3700,NA,2007\n', 'Adelie,Torgersen,41.1,17.6,182,3200,female,2007\n', 'Adelie,Torgersen,38.6,21.2,191,3800,male,2007\n', 'Adelie,Torgersen,34.6,21.1,198,4400,male,2007\n', 'Adelie,Torgersen,36.6,17.8,185,3700,female,2007\n', 'Adelie,Torgersen,38.7,19,195,3450,female,2007\n', 

In [ ]:
## Ejercicio: usando 'with', recorre el archivo línea por línea
## e imprime la especie de cada pingüino (sin acumular todo en una lista)

with open(ruta_archivo, "r") as archivo:
    next(archivo) # Salta el encabezado
    for linea in archivo:
        linea = archivo.readline()
        # tu código aquí (recuerda: la primera línea es el encabezado)
        datos = linea.strip().split(",")
        especie = datos[0]
        print(f"Especie: {especie}")

In [29]:
archivo = open(ruta_archivo, "r")
for linea in archivo:
  print(lineas_fas)

NameError: name 'lineas_fas' is not defined

In [22]:
print(type(archivo))

<class '_io.TextIOWrapper'>


In [23]:
print(archivo)

<_io.TextIOWrapper name='/content/penguins.csv' mode='r' encoding='utf-8'>


## Manejo de errores: `try` / `except`

¿Qué pasa si intentamos abrir un archivo que no existe, o cuyo nombre escribimos mal? Python lanza un error (`FileNotFoundError`) que detiene el programa. Con `try` / `except` podemos **capturar** ese error y decidir qué hacer en vez de que el programa se rompa.

In [30]:
try:
    with open("penguins_mal_escrito.csv", "r") as archivo:
        contenido = archivo.read()
except FileNotFoundError:
    print("El archivo no existe. Revisa el nombre o la ruta.")
except PermissionError:
    print("No tienes permisos para leer este archivo.")
except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")

El archivo no existe. Revisa el nombre o la ruta.


In [34]:
## Ejercicio: implementa el fragmento de código de la sección anterior
## (Imprime la especie de cada pingüino sin acumular todo en una lista)
## utilizando los bloques de Try / Except
try:
  with open(ruta_archivo, "r") as data:
    next(data)
    for linea in data:
      datos = linea.strip().split(",")
      especie = datos[0]
      print(f"Especie -> {especie}")
except FileNotFoundError:
    print("El archivo no existe. Revisa el nombre o la ruta.")
except PermissionError:
    print("No tienes permisos para leer este archivo.")
except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")



Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie -> Adelie
Especie ->

## Escribir archivos: modo `w`, modo `a`, `write()`, `writelines()`

* **`write(texto)`**: escribe un string en el archivo.
* **`writelines(lista)`**: escribe una lista de strings, uno tras otro (no agrega saltos de línea automáticamente).
* Modo **`"w"`**: si el archivo existe, **borra todo su contenido** antes de escribir. Úsalo con cuidado.
* Modo **`"a"`**: agrega al final del archivo sin borrar lo anterior.

In [35]:
## Ejemplo: filtrar solo los pingüinos de la especie Adelie
## y escribirlos en un archivo nuevo

with open(ruta_archivo, "r") as archivo:
  encabezado = archivo.readline()
  datos_filtrados = []
  for line in archivo:
    if line.startswith("Adelie"):
      datos_filtrados.append(line)

print(datos_filtrados[:6])

with open("adelie.csv", "w") as salida:
    salida.write(encabezado)
    salida.writelines(datos_filtrados)

print(f"Se guardaron {len(datos_filtrados)} registros de pingüinos Adelie en adelie.csv")


['Adelie,Torgersen,39.1,18.7,181,3750,male,2007\n', 'Adelie,Torgersen,39.5,17.4,186,3800,female,2007\n', 'Adelie,Torgersen,40.3,18,195,3250,female,2007\n', 'Adelie,Torgersen,NA,NA,NA,NA,NA,2007\n', 'Adelie,Torgersen,36.7,19.3,193,3450,female,2007\n', 'Adelie,Torgersen,39.3,20.6,190,3650,male,2007\n']
Se guardaron 152 registros de pingüinos Adelie en adelie.csv


```
## Ejercicio (modo append): recorre el archivo original y ve agregando,
## línea por línea, un log en 'log_sin_sexo.txt' cada vez que encuentres
## un pingüino sin el dato de 'sex' (campo vacío)
## Tip: abre 'log_sin_sexo.txt' en modo "w" antes del ciclo,
## y luego usa "a" dentro del ciclo — piensa cuál conviene más y por qué
```

In [42]:
log_file_name = 'log_sin_sexo_2.txt'

# Abre el archivo original para lectura
with open(ruta_archivo, "r") as infile:
    header = next(infile) # Ignora la línea del encabezado
    # Abre el archivo de log en modo escritura ('w') para asegurar que se inicia vacío
    # y lo mantendrá abierto para escribir durante el ciclo.
    with open(log_file_name, "a") as logfile:
        logfile.write(header)
        for line_num, line in enumerate(infile, 2): # Empieza a contar desde la línea 2 (después del encabezado)
            fields = line.strip().split(",")
            # La columna 'sex' es la séptima (índice 6)
            # Verificamos si hay suficientes campos y si el campo de sexo está vacío o es 'NA'
            if len(fields) > 6 and (fields[6].strip() == 'NA' or fields[6].strip() == ''):
                # Escribe la línea completa (incluyendo el salto de línea original) en el archivo de log
                logfile.write(f"Línea {line_num}: {line}")

print(f"Se ha generado '{log_file_name}' con los registros de pingüinos sin dato de sexo.")
print("Revisa el archivo para ver los detalles.")

Se ha generado 'log_sin_sexo_2.txt' con los registros de pingüinos sin dato de sexo.
Revisa el archivo para ver los detalles.


## Escribir archivos a partir de diccionarios

Muchas veces el resultado de un análisis no es una lista de líneas, sino un **diccionario**: por ejemplo, un conteo o un resumen. Veamos cómo pasar de un diccionario a un archivo de texto.

In [ ]:
## Ejemplo: contar cuántos pingüinos hay por especie usando un diccionario

conteo_especies = {}

## Lee todas las líneas del archivo y almacenalas en la lista ´lineas´
with open(ruta_archivo, "r") as archivo:
    lineas = archivo.

for linea in lineas[1:]:
    especie = linea.split(",")[0]
    conteo_especies[especie] = conteo_especies.get(especie, 0) + 1

print(conteo_especies)

In [ ]:
## Ahora escribimos ese diccionario a un archivo de texto,
## una línea por cada especie

with open("resumen_especies.txt", "w") as archivo:
    for especie, cantidad in conteo_especies.items():
        archivo.write(f"{especie}: {cantidad} pingüinos\n")

# Verifiquemos el resultado
with open("resumen_especies.txt", "r") as archivo:
    print(archivo.read())

## Ejercicio integrador: de FASTA a CSV

Tienes un archivo FASTA con varias secuencias de nucleótidos.
Tu tarea es leerlo y convertirlo en un archivo CSV con dos columnas: **header y secuencia**.


Recuerda: en un FASTA, cada secuencia puede venir partida en varias líneas. Tu código debe unir esas líneas correctamente antes de escribir el CSV.

In [43]:
from google.colab import files

uploaded = files.upload()

```
## -------------------------------------------------------------
# PASO 1: Define una función llamada 'leer_fasta' que reciba
## la ruta de un archivo y regrese un diccionario donde:
##   - cada llave sea un header (sin el símbolo '>')
##   - cada valor sea la secuencia completa (ya unida, sin saltos de línea)

## Dentro de la función necesitas:
##   a) Abrir el archivo con 'with' y recorrerlo línea por línea
##   b) Ignorar las líneas vacías
##   c) Si la línea empieza con '>' -> es un header nuevo.
##      Antes de reemplazar el header actual, guarda en el diccionario
##      el header y la secuencia que ya llevabas acumulada
##      (¡ojo! el primer header del archivo no tiene nada que guardar todavía)
##   d) Si la línea NO empieza con '>' -> es parte de la secuencia.
##      Agrégala a una lista de líneas de secuencia del header actual
##   e) Cuando el ciclo termine, no olvides guardar el último header
##      y su secuencia (el ciclo ya no lo va a hacer por ti)
##
## Tip: usa "".join(lista_de_lineas) para unir las líneas de secuencia
## en un solo string, en vez de concatenar con '+' en cada vuelta del ciclo


## -------------------------------------------------------------
## PASO 2: Usa tu función para leer 'secuencias.fasta' y guarda
## el resultado en una variable

## -------------------------------------------------------------
## PASO 3: Escribe el resultado en un archivo CSV llamado
## 'secuencias_procesadas.csv', con:
##   - una primera línea de encabezado: header,secuencia
##   - una línea por cada secuencia encontrada
##
## Tip: recorre el diccionario con .items() para obtener
## header y secuencia al mismo tiempo

## -------------------------------------------------------------
## PASO 4 (verificación): abre 'secuencias_procesadas.csv' en modo
## lectura e imprime su contenido, para confirmar que se escribió
## correctament
```

In [45]:
# --- 1. Nombres de los archivos con los que vamos a trabajar ---

archivo_fasta_entrada = "/content/secuencias_pyladiescuerna.fasta"
archivo_salida = "secuencias_procesadas.csv"

# --- 2. Función para leer el archivo multifasta ---
def leer_fasta(ruta_archivo):
    secuencias = {}
    header_actual = None
    lineas_secuencia_actual = []

##   a) Abrir el archivo con 'with' y recorrerlo línea por línea
    with open(ruta_archivo, 'r') as archivo:
        for linea in archivo:
            linea = linea.strip() # Elimina salto de línea al final
            ##   b) Ignorar las líneas vacías
            if not linea:  # ignorar líneas vacías
                continue

            if linea.startswith(">"):
                ##   c) Si la línea empieza con '>' -> es un header nuevo.
                if header_actual and lineas_secuencia_actual:  # guardar la secuencia anterior
                    secuencias[header_actual] = "".join(lineas_secuencia_actual)

                header_actual = linea[1:]  # eliminar el '>' del header
                lineas_secuencia_actual = []
            else:
                ##   d) Si la línea NO empieza con '>' -> es parte de la secuencia.
                # añadir esta línea a la secuencia que se está armando en la lista de secuencia
                lineas_secuencia_actual.append(linea)

##   e) Cuando el ciclo termine, no olvides guardar el último header
##      y su secuencia (el ciclo ya no lo va a hacer por ti)

        # guardar la última secuencia después de terminar el ciclo
        if header_actual and lineas_secuencia_actual:
            secuencias[header_actual] = "".join(lineas_secuencia_actual)

    return secuencias

# --- 3. Procesar el archivo y escribir la salida ---
try:

    ## PASO 2: Usa tu función para leer 'secuencias.fasta' y guarda
    ## el resultado en una variable
    secuencias_encontradas = leer_fasta(archivo_fasta_entrada)

    ## PASO 3: Escribe el resultado en un archivo CSV llamado
    ## 'secuencias_procesadas.csv', con:
    ##   - una primera línea de encabezado: header,secuencia
    ##   - una línea por cada secuencia encontrada
    with open(archivo_salida, 'w') as salida:
        salida.write("IDs,Secuencia\n")
        for header, secuencia in secuencias_encontradas.items():
            salida.write(f"{header},{secuencia}\n")

    ## PASO 4 (verificación): abre 'secuencias_procesadas.csv' en modo
  ## lectura e imprime su contenido
    print(f"Datos procesados y guardados en '{archivo_salida}'.")
    print(f"\nContenido de '{archivo_salida}':")
    with open(archivo_salida, 'r') as salida:
        print(salida.read())

except FileNotFoundError:
    print(f"Error: el archivo '{archivo_fasta_entrada}' no se encontró.")
except Exception as e:
    print(f"Ocurrió un error: {e}")

Datos procesados y guardados en 'secuencias_procesadas.csv'.

Contenido de 'secuencias_procesadas.csv':
IDs,Secuencia
GCA_003044255.1_1933993-1934473,GTTCACTGCCGCACAGGCAGCTTAGAAATCTGGATCAATATCAATATCTACTTCAGCAAAGTTCACTGCCGCACAGGCAGCTTAGAAATAGAAGCGCATGCAGGCAATGGGTGAAGACATGTTCACTGCCGCACAGGCAGCTTAGAAAAGAATCATAGCCTTCTACATTGTGCAGCGGATGTTCACTGCCGCACAGGCAGCTTAGAAATGCATCGTCATTAGCTGGCAATTCAGGCTTAGGTTCACTGCCGCACAGGCAGCTTAGAAATGCCATATCGGCGGCGGCATCAATGTAATTTTGTTCACTGCCGCACAGGCAGCTTAGAAAATTTTAGGAATAAATCATATATTTGTAGTTAAGTTCACTGCCGCACAGGCAGCTTAGAAACAATTGAGATTACTTTGGCTGGTTGTCGAAACGTTCACTGCCGCACAGGCAGCTTAGAAAAATCATCGACATTAATACCGATGAACTTTATGGTTCACTGCCGCACAGGCAGCTTAGATG
GCA_006401635.1_8369628-8369845,GTTTCAACCCACGCTCCTCTCGTTCACGAGAAGCGACGCACCTGCTGGGCCAGCGGCGCTACGCCCTGCATGTTTCAACCCACGCTCCTTGCGTTCACGAGGAGCGACGCCCGGGTTGCGCAGCGCGCCCTCGAGGCCCGTGCGGTTTCAACCCACGCTCCTCGCGTTCACGAGGAGCGACTTTCACTTCCACGTCAGCCACGACGGGACAATCCGCGTTTCAATCCACGCTCCTCGCGTTCACGAGGAGCGAC
GCA_006401635.1_8665570-8666226,GTGCTCAACGCCTTTCGGCATC

##  Cierre

Hoy vimos cómo abrir, leer, escribir y manejar errores de archivos en Python, usando `penguins.csv` y un archivo multiFASTA como ejemplo real.